In [1]:
# Install the required LangChain packages
!pip install -q langchain langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00


In [2]:
import os
import getpass

print("Enter your Groq API Token:")
os.environ["GROQ_API_KEY"] = getpass.getpass()

print("\nEnter your LangSmith API Key (starts with lsv2_...):")
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()

# Enable LangSmith Tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "AI_Resume_Screener"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

print("\nEnvironment and Tracing setup complete!")

Enter your Groq API Token:
··········

Enter your LangSmith API Key (starts with lsv2_...):
··········

Environment and Tracing setup complete!


In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser

# 1. Define the Prompt (Forces JSON output)
eval_template = """
You are an expert AI technical recruiter. Your task is to evaluate a candidate's resume against a Job Description.

JOB DESCRIPTION:
{job_description}

CANDIDATE RESUME:
{resume}

INSTRUCTIONS:
1. Extract the candidate's Skills, Experience, and Tools.
2. Match them against the Job Description.
3. Assign a Fit Score from 0 to 100 based on how well they match.
4. Explain exactly WHY you gave this score.
5. DO NOT assume or hallucinate skills that are not explicitly written in the resume.

Respond STRICTLY in the following JSON format:
{{
    "extracted_skills": ["skill1", "skill2"],
    "experience_summary": "Brief summary.",
    "match_analysis": "What matches and what is missing.",
    "score": 85,
    "explanation": "Detailed reason."
}}
"""

resume_prompt = PromptTemplate(
    input_variables=["job_description", "resume"],
    template=eval_template
)

# 2. Define the Chain
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
chain = resume_prompt | llm | JsonOutputParser()

In [4]:
import json

job_description = """
Role: Junior Data Scientist
Requirements:
- 1+ years of experience in Data Science or Analytics.
- Strong proficiency in Python, SQL, and Machine Learning.
- Experience with Generative AI tools (LangChain, OpenAI) is a massive plus.
- Good communication skills.
"""

resumes = {
    "Strong Candidate": """
        Name: Alice Data
        Experience: 2 years as a Junior Data Scientist at TechCorp.
        Skills: Python, SQL, Machine Learning, Deep Learning.
        Projects: Built an internal chatbot using LangChain and LLMs.
    """,
    "Average Candidate": """
        Name: Bob Analyst
        Experience: 1.5 years as a Data Analyst.
        Skills: Python, SQL, Excel, Tableau.
        Projects: Created sales dashboards and wrote SQL queries.
    """,
    "Weak Candidate": """
        Name: Charlie Frontend
        Experience: 3 years as a Web Developer.
        Skills: JavaScript, React, HTML, CSS.
        Projects: Built e-commerce websites.
    """
}

# Run the evaluation
print("Starting AI Resume Evaluation...\n")

for candidate_type, resume_text in resumes.items():
    print(f"--- Evaluating: {candidate_type} ---")

    # We tag the run so it shows up neatly in LangSmith!
    result = chain.invoke(
        {"job_description": job_description, "resume": resume_text},
        config={"tags": [candidate_type.replace(" ", "_").lower()]}
    )

    # Print the beautiful JSON output
    print(json.dumps(result, indent=4))
    print("\n" + "="*60 + "\n")

Starting AI Resume Evaluation...

--- Evaluating: Strong Candidate ---
{
    "extracted_skills": [
        "Python",
        "SQL",
        "Machine Learning",
        "Deep Learning"
    ],
    "experience_summary": "2 years as a Junior Data Scientist at TechCorp",
    "match_analysis": "The candidate's skills in Python, SQL, and Machine Learning match the job requirements. Additionally, their experience with LangChain, a Generative AI tool, is a significant plus. However, the job description requires good communication skills, which are not mentioned in the resume.",
    "score": 85,
    "explanation": "I gave this score because the candidate has a strong technical background that aligns with the job requirements, including experience with a Generative AI tool. However, the lack of explicitly mentioned communication skills prevents a perfect score, as this is an essential requirement for the Junior Data Scientist role."
}


--- Evaluating: Average Candidate ---
{
    "extracted_skill